# RAG untuk Data Terstruktur

Bangun alur tanya-jawab yang membaca data PostgreSQL: skema → SQL dari OpenAI → eksekusi read-only → jawaban berbasis hasil.

**Waktu:** 75–90 menit · **Hasil:** pipeline yang bisa menjawab tiga pertanyaan bisnis dari database Neon/PostgreSQL.


## Alur lab

Kita memakai tabel pesanan, pengiriman, dan karyawan sintetis. Berbeda dari RAG dokumen, retrieval di sini memakai SQL untuk mengambil baris yang tepat.

1. Siapkan database dan akses read-only.
2. Periksa tabel lalu kirim skema ringkas ke model.
3. Tinjau SQL buatan model sebelum aplikasi menjalankannya.
4. Kirim hasil query ke model untuk jawaban singkat.


## 1 — Siapkan lingkungan

**Tujuan:** jalankan notebook dan dependensinya dari folder project ini.

Windows PowerShell:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
jupyter lab 18_rag_structured_data.ipynb
```

macOS/Linux:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter lab 18_rag_structured_data.ipynb
```

**Checkpoint:** Jupyter membuka notebook ini dengan kernel Python 3.11 atau yang lebih baru.


## 2 — Data dan akses read-only

**Tujuan:** gunakan data sintetis untuk tiga pertanyaan lab.

File setup.sql di folder ini membuat ulang schema m18_rag_structured_data. Jalankan hanya pada database lab khusus. Skrip membatasi perubahan ke schema tersebut dan membuat view employee_directory tanpa salary.

Jika instruktur sudah memberi URL database lab, lewati setup dan gunakan URL itu. Untuk database sendiri, jalankan setup.sql, siapkan login read-only bernama m18_rag_reader, lalu gunakan template setup_reader_role.sql.example. Salin template ke setup_reader_role.local.sql, isi password lokal, dan jalankan sekali.

> **Catatan seed:** menjalankan setup.sql akan mengganti tabel demo di schema lab.

**Checkpoint:** gunakan URL m18_rag_reader, bukan URL pemilik database.


In [4]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd().resolve()
NOTEBOOK_PATH = PROJECT_DIR / "18_rag_structured_data.ipynb"
SETUP_SQL_PATH = PROJECT_DIR / "setup.sql"
if not NOTEBOOK_PATH.is_file() or not SETUP_SQL_PATH.is_file():
    raise FileNotFoundError(
        "Jalankan Jupyter dari folder project_v2 yang berisi notebook dan setup.sql."
    )

DB_SCHEMA = "m18_rag_structured_data"
load_dotenv(PROJECT_DIR / ".env", override=False)
print("Schema lab:", DB_SCHEMA)
print("Setup data tersedia:", SETUP_SQL_PATH.is_file())


Schema lab: m18_rag_structured_data
Setup data tersedia: True


## 3 — Masukkan kredensial dengan aman

**Tujuan:** hubungkan notebook tanpa menulis rahasia ke cell atau output.

Buat file .env dari .env.example di folder ini, atau atur environment variables. Notebook membaca RAG_DATABASE_URL dan OPENAI_API_KEY. Jika nilainya belum tersedia, notebook akan meminta input tersembunyi.

> **Keamanan:** jangan bagikan file .env atau setup_reader_role.local.sql.


In [5]:
import getpass
import os

DATABASE_URL = (
    os.environ.get("RAG_DATABASE_URL")
    or os.environ.get("DATABASE_URL")
    or getpass.getpass("URL PostgreSQL role read-only: ")
)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass(
    "OPENAI_API_KEY: "
)
MODEL = os.environ.get("AGENT_MODEL", "gpt-4o-mini")

if not DATABASE_URL or not OPENAI_API_KEY:
    raise ValueError("URL database dan OPENAI_API_KEY wajib diisi.")
if DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = "postgresql://" + DATABASE_URL.removeprefix("postgres://")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("Kredensial tersedia; tidak ada rahasia yang ditampilkan.")
print("Model:", MODEL)


Kredensial tersedia; tidak ada rahasia yang ditampilkan.
Model: gpt-5.6-luna


## 4 — Hubungkan ke PostgreSQL

**Tujuan:** pastikan database nyata dapat diakses dan hanya objek lab yang terlihat.

**Checkpoint:** periksa database, role, timeout, serta tabel. Login harus m18_rag_reader dan schema harus m18_rag_structured_data.


In [ ]:
!pip install langchain_community

In [7]:
from sqlalchemy import create_engine, text
from langchain_community.utilities import SQLDatabase

# -======= ini bagian untuk connect langchainm buat object databae
TABLES = ["orders", "order_items", "shipments", "departments", "employee_directory"]
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
)
db = SQLDatabase(
    engine=engine,
    schema=DB_SCHEMA,
    include_tables=TABLES,
    view_support=True,
    sample_rows_in_table_info=0,
)

with engine.connect() as connection:
    connection_info = connection.execute(
        text(
            "SELECT current_database() AS database, "
            "current_user AS role, "
            "current_setting('statement_timeout') AS statement_timeout"
        )
    ).mappings().one()

available_tables = db.get_usable_table_names()
missing_tables = sorted(set(TABLES) - set(available_tables))
print(dict(connection_info))
print("Tabel lab:", available_tables)
# Bypass lokal: role check m18_rag_reader dilewati untuk testing dengan akun lokal.
if missing_tables:
    raise RuntimeError(
        "Objek lab belum lengkap: " + ", ".join(missing_tables)
        + ". Pastikan schema dan view lab sudah disiapkan."
    )


/var/folders/cx/330dqlbn5w7_rk__cddfhj3m0000gq/T/ipykernel_23997/4202863225.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


{'database': 'm18_rag_structured_data', 'role': 'postgres', 'statement_timeout': '0'}
Tabel lab: ['departments', 'employee_directory', 'order_items', 'orders', 'shipments']


## 5 — Amati skema yang akan dipakai model

**Tujuan:** lihat tabel, kolom, dan tipe data sebelum SQL dibuat. Baris contoh tidak dikirim ke prompt.

**Checkpoint:** skema mencakup orders, shipments, order_items, dan employee_directory, tanpa kolom salary.


In [8]:
schema_text = db.get_table_info(TABLES)
print(schema_text)



CREATE TABLE m18_rag_structured_data.departments (
	id INTEGER NOT NULL, 
	name TEXT NOT NULL, 
	CONSTRAINT departments_pkey PRIMARY KEY (id)
)


CREATE TABLE m18_rag_structured_data.employee_directory (
	id INTEGER, 
	name TEXT, 
	department_id INTEGER, 
	manager_id INTEGER
)


CREATE TABLE m18_rag_structured_data.order_items (
	id INTEGER NOT NULL, 
	order_id INTEGER, 
	product TEXT NOT NULL, 
	qty INTEGER NOT NULL, 
	price INTEGER NOT NULL, 
	CONSTRAINT order_items_pkey PRIMARY KEY (id), 
	CONSTRAINT order_items_order_id_fkey FOREIGN KEY(order_id) REFERENCES m18_rag_structured_data.orders (id)
)


CREATE TABLE m18_rag_structured_data.orders (
	id INTEGER NOT NULL, 
	customer_id INTEGER NOT NULL, 
	status TEXT NOT NULL, 
	total INTEGER NOT NULL, 
	created_at DATE NOT NULL, 
	CONSTRAINT orders_pkey PRIMARY KEY (id)
)


CREATE TABLE m18_rag_structured_data.shipments (
	id INTEGER NOT NULL, 
	order_id INTEGER, 
	carrier TEXT NOT NULL, 
	status TEXT NOT NULL, 
	shipped_at DATE, 
	delive

## 6 — Bangun penyusun SQL dengan OpenAI

**Tujuan:** suntikkan skema runtime ke system prompt. Model hanya mengusulkan satu pernyataan `SELECT`; aplikasi tetap memegang koneksi database.

Setiap pertanyaan memanggil API OpenAI dua kali: untuk SQL dan jawaban. Perhatikan penggunaan API. Tidak ada mock pada notebook ini.


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
# INI KALO PAKE MINI 40 PAKE TEMPERATURE || KALO PAKE LUNA NGGK USAH DIPAKE IS OKE
# llm = ChatOpenAI(model=MODEL, temperature=0)
llm = ChatOpenAI(model=MODEL)
sql_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Anda menyusun SQL PostgreSQL untuk aplikasi analitik.\n"
            "Gunakan hanya tabel dan kolom yang tercantum pada skema.\n"
            "Kembalikan tepat satu query SELECT tanpa markdown atau penjelasan.\n"
            "Jangan meminta atau menampilkan data pribadi yang tidak diperlukan.\n\n"
            "Skema database:\n{schema}",
        ),
        ("human", "Pertanyaan: {question}"),
    ]
)
sql_chain = sql_prompt | llm | StrOutputParser()

import re


def clean_sql(text: str) -> str:
    """Hapus pembungkus teks umum sebelum SQL diperiksa."""
    candidate = text.strip()
    candidate = re.sub(r"^```(?:sql)?\s*", "", candidate, flags=re.IGNORECASE)
    candidate = re.sub(r"\s*```$", "", candidate).strip()
    candidate = re.sub(r"^SQLQuery:\s*", "", candidate, flags=re.IGNORECASE)
    return candidate


def generate_sql(question: str) -> str:
    """Minta model menyusun SQL dari pertanyaan dan skema yang tersedia."""
    raw_sql = sql_chain.invoke({"schema": schema_text, "question": question})
    return clean_sql(raw_sql)

print("OpenAI client siap; panggilan API terjadi saat generate_sql dipanggil.")


OpenAI client siap; panggilan API terjadi saat generate_sql dipanggil.


## 7 — Minta SQL untuk tiga pertanyaan

**Tujuan:** bandingkan pertanyaan analitik, status transaksi, dan relasi karyawan.

Baca SQL yang diusulkan sebelum menjalankan cell eksekusi. Apakah model memilih tabel dan kondisi yang tepat?


In [11]:
QUESTIONS = {
    "paid_orders": "Berapa total nilai order yang statusnya paid?",
    "shipment_1004": "Apa status dan kurir pengiriman untuk order 1004?",
    "manager_andi": "Siapa manajer dari karyawan bernama Andi Pratama?",
}

cases = {}
for name, question in QUESTIONS.items():
    sql = generate_sql(question)
    cases[name] = {"question": question, "sql": sql}
    print(f"--- {name} ---")
    print(sql, "\n")


--- paid_orders ---
SELECT COALESCE(SUM(total), 0) AS total_nilai_order
FROM m18_rag_structured_data.orders
WHERE status = 'paid'; 

--- shipment_1004 ---
SELECT status, carrier
FROM m18_rag_structured_data.shipments
WHERE order_id = 1004; 

--- manager_andi ---
SELECT manager.name AS manager_name
FROM m18_rag_structured_data.employee_directory AS employee
LEFT JOIN m18_rag_structured_data.employee_directory AS manager
  ON manager.id = employee.manager_id
WHERE employee.name = 'Andi Pratama'; 



**Checkpoint — tinjau SQL sebelum lanjut.** Pastikan setiap query hanya membaca data, memakai tabel lab, dan menjawab pertanyaan yang diminta. Role read-only dan guardrail berikut menjadi lapisan perlindungan tambahan.


In [8]:
!pip install sqlparse

  Using cached sqlparse-0.6.0-py3-none-any.whl.metadata (6.0 kB)
Using cached sqlparse-0.6.0-py3-none-any.whl (50 kB)


In [12]:
import sqlparse


def validate_select(sql: str) -> str:
    """Terima tepat satu SELECT; role dan transaksi read-only tetap menjadi batas utama."""
    candidate = sqlparse.format(sql, strip_comments=True).strip()
    statements = [statement for statement in sqlparse.parse(candidate) if str(statement).strip()]
    if len(statements) != 1:
        raise ValueError("Query ditolak: hanya satu pernyataan SQL yang diizinkan.")

    first_token = statements[0].token_first(skip_cm=True, skip_ws=True)
    if first_token is None or first_token.normalized.upper() != "SELECT":
        raise ValueError("Query ditolak: lab ini hanya menerima SELECT, tanpa CTE atau perintah tulis.")

    return candidate.rstrip(";").strip()


def execute_readonly(sql: str, max_rows: int = 50) -> tuple[list[str], list[tuple]]:
    """Jalankan SELECT dalam transaksi read-only dan batasi baris yang dikembalikan."""
    if not 1 <= max_rows <= 50:
        raise ValueError("max_rows harus berada antara 1 dan 50.")

    safe_sql = validate_select(sql)
    bounded_sql = f"SELECT * FROM ({safe_sql}) AS bounded_result LIMIT {max_rows}"
    with engine.connect() as connection:
        with connection.begin():
            connection.exec_driver_sql("SET TRANSACTION READ ONLY")
            connection.exec_driver_sql(f"SET LOCAL search_path TO {DB_SCHEMA}")
            result = connection.execute(text(bounded_sql))
            columns = list(result.keys())
            rows = [tuple(row) for row in result.fetchall()]
    return columns, rows

print("Guardrail siap: satu SELECT, transaksi read-only, maksimal 50 baris, timeout 5 detik.")


Guardrail siap: satu SELECT, transaksi read-only, maksimal 50 baris, timeout 5 detik.


## 8 — Validasi usulan SQL

**Tujuan:** jalankan pemeriksaan satu `SELECT` sebelum menyentuh database.

**Checkpoint:** jika model membuat perintah tulis atau beberapa statement, notebook berhenti dengan pesan penolakan. Periksa juga isi SQL secara manual.


In [13]:
validated_sql = {}
for name, case in cases.items():
    validated_sql[name] = validate_select(case["sql"])
    print(f"{name}: lolos pemeriksaan satu SELECT")


paid_orders: lolos pemeriksaan satu SELECT
shipment_1004: lolos pemeriksaan satu SELECT
manager_andi: lolos pemeriksaan satu SELECT


## 8a — Pastikan guardrail menolak query berbahaya

**Checkpoint:** uji tanpa mengirim query ke database. Perintah tulis dan dua statement harus ditolak.


In [14]:
blocked_queries = [
    "DELETE FROM m18_rag_structured_data.orders",
    "SELECT 1; DROP TABLE m18_rag_structured_data.orders",
]
for query in blocked_queries:
    try:
        validate_select(query)
    except ValueError as error:
        print("Ditolak:", error)
    else:
        raise RuntimeError("Guardrail gagal menolak query yang tidak aman.")


Ditolak: Query ditolak: lab ini hanya menerima SELECT, tanpa CTE atau perintah tulis.
Ditolak: Query ditolak: hanya satu pernyataan SQL yang diizinkan.


## 9 — Jalankan query dengan batas aman

**Tujuan:** ambil baris dan kolom dari database. Aplikasi menjalankan SQL, bukan model.

**Checkpoint:** untuk seed yang disediakan, nilai order berstatus `paid` berjumlah **Rp1.360.000**; shipment order `1004` berstatus `in_transit`; manajer Andi adalah **Dewi Lestari**. Nilai berubah jika data berubah.


In [15]:
results = {}
for name, case in cases.items():
    columns, rows = execute_readonly(validated_sql[name])
    results[name] = {
        **case,
        "sql": validated_sql[name],
        "columns": columns,
        "rows": rows,
    }
    print(f"--- {name} ---")
    print("Kolom:", columns)
    print("Baris:", rows, "\n")


--- paid_orders ---
Kolom: ['total_nilai_order']
Baris: [(1360000,)] 

--- shipment_1004 ---
Kolom: ['status', 'carrier']
Baris: [('in_transit', 'JNE')] 

--- manager_andi ---
Kolom: ['manager_name']
Baris: [('Dewi Lestari',)] 



## 10 — Rangkai jawaban yang grounded

**Tujuan:** kirim pertanyaan dan hasil query yang sudah dibatasi ke model. Jawaban harus bersumber dari baris tersebut; jika tidak ada baris, model harus menyatakan bahwa data tidak ditemukan.


In [18]:
import json

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Jawab singkat dalam Bahasa Indonesia hanya dari hasil query yang diberikan. "
            "Jangan menambah fakta. Jika hasil kosong, katakan data tidak ditemukan.",
        ),
        (
            "human",
            "Pertanyaan: {question}\nSQL: {sql}\nKolom: {columns}\n"
            "Hasil query (JSON): {rows_json}",
        ),
    ]
)
answer_chain = answer_prompt | llm | StrOutputParser()

for name, result in results.items():
    rows_json = json.dumps(result["rows"], ensure_ascii=False, default=str)
    result["answer"] = answer_chain.invoke(
        {
            "question": result["question"],
            "sql": result["sql"],
            "columns": result["columns"],
            "rows_json": rows_json,
        }
    )
    print(f"--- {name} ---")
    print("SQL:", result["sql"])
    print("Baris:", result["rows"])
    print("Jawaban:", result["answer"], "\n")


--- paid_orders ---
SQL: SELECT COALESCE(SUM(total), 0) AS total_nilai_order
FROM m18_rag_structured_data.orders
WHERE status = 'paid'
Baris: [(1360000,)]
Jawaban: Total nilai order dengan status **paid** adalah **1.360.000**. 

--- shipment_1004 ---
SQL: SELECT status, carrier
FROM m18_rag_structured_data.shipments
WHERE order_id = 1004
Baris: [('in_transit', 'JNE')]
Jawaban: Status pengiriman order 1004 adalah **in_transit** dan kurirnya **JNE**. 

--- manager_andi ---
SQL: SELECT manager.name AS manager_name
FROM m18_rag_structured_data.employee_directory AS employee
LEFT JOIN m18_rag_structured_data.employee_directory AS manager
  ON manager.id = employee.manager_id
WHERE employee.name = 'Andi Pratama'
Baris: [('Dewi Lestari',)]
Jawaban: Manajer Andi Pratama adalah Dewi Lestari. 



## Edge case — tidak ada baris yang cocok

**Checkpoint:** query order yang tidak ada. Hasil kosong harus tetap menghasilkan jawaban yang jujur dan grounded.


In [19]:
empty_question = "Apa status order 999999?"
empty_sql = "SELECT id, status FROM m18_rag_structured_data.orders WHERE id = 999999"
empty_columns, empty_rows = execute_readonly(empty_sql)
empty_answer = answer_chain.invoke(
    {
        "question": empty_question,
        "sql": empty_sql,
        "columns": empty_columns,
        "rows_json": json.dumps(empty_rows, ensure_ascii=False, default=str),
    }
)
print("Baris:", empty_rows)
print("Jawaban:", empty_answer)


Baris: []
Jawaban: Data order 999999 tidak ditemukan.


## Apa yang keluar dari database?

Saat SQL dibuat, OpenAI menerima pertanyaan dan skema terpilih. Saat jawaban dirangkai, OpenAI juga menerima SQL dan hasil baris query. Di sini data sintetis, tetapi pola yang sama dapat mengirim data sensitif dari sistem nyata.

**Diskusikan:** kolom apa yang perlu dikecualikan atau dianonimkan pada kasus produksi?


# Complete Chain

Satu LCEL pipeline merangkai scan schema → pertanyaan → SQL → tinjauan → query read-only → jawaban.

Tahap pemodelan memakai sintaks LangChain LCEL. Persetujuan tetap menjadi gerbang sebelum database dibaca.

**Checkpoint:** baca SQL yang diusulkan. Eksekusi hanya berjalan setelah Anda mengetik y.


In [14]:
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough

# Rangkai tahap pemodelan dengan LCEL; validasi dan database tetap terlihat sebagai tahap sendiri.
complete_sql_generation = (
    sql_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_sql)
)

def scan_complete_schema(_: dict) -> dict:
    schema = db.get_table_info(TABLES)
    print("[1/5] Schema yang dibaca:")
    print(schema)
    return {"schema": schema}

def ask_complete_question(_: dict) -> str:
    question = input("[2/5] Tulis pertanyaan (Enter untuk contoh): ").strip()
    if not question:
        question = "Berapa total nilai order yang statusnya paid?"
    print("Pertanyaan:", question)
    return question

def review_complete_sql(state: dict) -> dict:
    safe_sql = validate_select(state["sql"])
    print("[3/5] SQL yang diusulkan:")
    print(safe_sql)
    print("Validasi: satu SELECT lolos.")
    confirmation = input(
        "Tinjau SQL di atas. Ketik y untuk menarik data dari database: "
    ).strip().lower()
    return {**state, "safe_sql": safe_sql, "approved": confirmation == "y"}

def pull_complete_rows(state: dict) -> dict:
    if not state["approved"]:
        return {**state, "columns": [], "rows": []}
    print("[4/5] Menjalankan query dalam transaksi read-only...")
    columns, rows = execute_readonly(state["safe_sql"])
    print("Kolom:", columns)
    print("Baris:", rows)
    return {**state, "columns": columns, "rows": rows}

def make_complete_answer_input(state: dict) -> dict:
    return {
        "question": state["question"],
        "sql": state["safe_sql"],
        "columns": state["columns"],
        "rows_json": json.dumps(state["rows"], ensure_ascii=False, default=str),
    }

def display_complete_answer(state: dict) -> dict:
    print("[5/5] Jawaban:", state["answer"])
    return state

approved_answer = RunnablePassthrough.assign(
    answer=RunnableLambda(make_complete_answer_input) | answer_chain
)
cancelled_answer = RunnablePassthrough.assign(
    answer=RunnableLambda(lambda _: "Eksekusi dibatalkan. SQL belum dikirim ke database.")
)

complete_chain = (
    RunnableLambda(scan_complete_schema)
    | RunnablePassthrough.assign(question=RunnableLambda(ask_complete_question))
    | RunnablePassthrough.assign(sql=complete_sql_generation)
    | RunnableLambda(review_complete_sql)
    | RunnableLambda(pull_complete_rows)
    | RunnableBranch((lambda state: state["approved"], approved_answer), cancelled_answer)
    | RunnableLambda(display_complete_answer)
)

complete_chain_result = complete_chain.invoke({})


[1/5] Schema yang dibaca:

CREATE TABLE m18_rag_structured_data.departments (
	id INTEGER NOT NULL, 
	name TEXT NOT NULL, 
	CONSTRAINT departments_pkey PRIMARY KEY (id)
)


CREATE TABLE m18_rag_structured_data.employee_directory (
	id INTEGER, 
	name TEXT, 
	department_id INTEGER, 
	manager_id INTEGER
)


CREATE TABLE m18_rag_structured_data.order_items (
	id INTEGER NOT NULL, 
	order_id INTEGER, 
	product TEXT NOT NULL, 
	qty INTEGER NOT NULL, 
	price INTEGER NOT NULL, 
	CONSTRAINT order_items_pkey PRIMARY KEY (id), 
	CONSTRAINT order_items_order_id_fkey FOREIGN KEY(order_id) REFERENCES m18_rag_structured_data.orders (id)
)


CREATE TABLE m18_rag_structured_data.orders (
	id INTEGER NOT NULL, 
	customer_id INTEGER NOT NULL, 
	status TEXT NOT NULL, 
	total INTEGER NOT NULL, 
	created_at DATE NOT NULL, 
	CONSTRAINT orders_pkey PRIMARY KEY (id)
)


CREATE TABLE m18_rag_structured_data.shipments (
	id INTEGER NOT NULL, 
	order_id INTEGER, 
	carrier TEXT NOT NULL, 
	status TEXT NOT NULL, 


Pertanyaan: Berapa total nilai order yang statusnya paid?
[3/5] SQL yang diusulkan:
SELECT SUM(total) AS total_paid_orders FROM m18_rag_structured_data.orders WHERE status = 'paid'
Validasi: satu SELECT lolos.
[4/5] Menjalankan query dalam transaksi read-only...
Kolom: ['total_paid_orders']
Baris: [(1360000,)]
[5/5] Jawaban: Total nilai order yang statusnya paid adalah 1.360.000.


## Tantangan — penjualan per produk

**Tujuan:** terapkan pola pada pertanyaan yang memerlukan `JOIN`, filter status, dan agregasi.

Tanyakan nilai penjualan per produk dari order berstatus `paid`, urut dari terbesar. Prediksi tabel, join, dan rumus agregasinya sebelum menjalankan cell berikut.


In [15]:
challenge_question = (
    "Berapa nilai penjualan per produk dari order yang berstatus paid, "
    "urutkan dari nilai terbesar?"
)
challenge_sql = generate_sql(challenge_question)
print("Tinjau SQL tantangan ini sebelum lanjut:\n")
print(challenge_sql)


Tinjau SQL tantangan ini sebelum lanjut:

SELECT product, SUM(qty * price) AS total_sales
FROM m18_rag_structured_data.order_items oi
JOIN m18_rag_structured_data.orders o ON oi.order_id = o.id
WHERE o.status = 'paid'
GROUP BY product
ORDER BY total_sales DESC;


**Checkpoint tantangan:** cek bahwa query menggabungkan `orders` dengan `order_items`, memfilter `paid`, lalu mengelompokkan hasil per produk. Jalankan cell berikut hanya setelah memeriksa SQL tersebut.


In [16]:
challenge_columns, challenge_rows = execute_readonly(challenge_sql)
challenge_answer = answer_chain.invoke(
    {
        "question": challenge_question,
        "sql": validate_select(challenge_sql),
        "columns": challenge_columns,
        "rows_json": json.dumps(challenge_rows, ensure_ascii=False, default=str),
    }
)
print("Kolom:", challenge_columns)
print("Baris:", challenge_rows)
print("Jawaban:", challenge_answer)


Kolom: ['product', 'total_sales']
Baris: [('Webcam', 560000), ('Keyboard', 450000), ('Cable USB', 230000), ('Mouse', 120000)]
Jawaban: 1. Webcam: 560000
2. Keyboard: 450000
3. Cable USB: 230000
4. Mouse: 120000


## Ringkasan

- RAG database menyusun SQL dari skema, lalu mengambil baris yang relevan.
- Aplikasi memvalidasi dan menjalankan query; model tidak memegang koneksi database.
- Jawaban hanya sekuat query dan hasil yang menjadi buktinya.
- **Refleksi:** sebutkan satu kontrol akses dan satu perlindungan privasi yang wajib dibawa ke produksi.
